# MediCS — Colab GPU Runner

Run cells in order. Each cell is independent — re-run any cell to check scores.

**Pipeline:** Setup → Inference → Judge → SFT → repeat → DPO → Eval → Transfer

In [ ]:
# Cell 1 — Setup (run once per session)
from google.colab import drive
drive.mount('/content/drive')

import sys, subprocess, os
sys.path.insert(0, '/content/drive/MyDrive/MediCS')
os.chdir('/content/drive/MyDrive/MediCS')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'transformers', 'peft', 'trl', 'datasets',
    'accelerate', 'bitsandbytes', 'sentence-transformers',
    'deep-translator', 'openai', 'huggingface_hub',
])

!nvidia-smi
print(f"\nWorking dir: {os.getcwd()}")
print("Setup complete.")

## Round 1 — Base Model Attack

In [ ]:
# Cell 2 — Round 1: Inference (base model, BASE prompt)
!python colab/run_inference.py \
    --checkpoint base --seed 42 \
    --input results/attacks/round_1/prompts.jsonl \
    --output results/attacks/round_1/responses.jsonl

In [ ]:
# Cell 3 — Round 1: Judge responses (GPT-5, runs locally but can run here too)
# NOTE: This calls the OpenAI API — costs ~$0.40
!python scripts/02_run_attack_round.py --round 1 --phase judge

In [ ]:
# Cell 4 — Round 1: Check ASR
from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_1/results.jsonl")
print(f"Round 1 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

In [ ]:
# Cell 5 — Round 1: Build SFT data + Train SFT
!python scripts/03_build_defense_data.py --rounds 1 --type sft
!python colab/train_sft.py --round 1

## Round 2 — Attack SFT Round 1

In [ ]:
# Cell 6 — Round 2: Generate attacks (locally, but can run here)
!python scripts/02_run_attack_round.py --round 2 --phase generate

In [ ]:
# Cell 7 — Round 2: Inference (SFT round 1 checkpoint, BASE prompt)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_1/final --seed 42 \
    --input results/attacks/round_2/prompts.jsonl \
    --output results/attacks/round_2/responses.jsonl

In [ ]:
# Cell 8 — Round 2: Judge + ASR
!python scripts/02_run_attack_round.py --round 2 --phase judge

from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_2/results.jsonl")
print(f"\nRound 2 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

In [ ]:
# Cell 9 — Round 2: Build SFT data + Train SFT (continuing from round 1)
!python scripts/03_build_defense_data.py --rounds 1,2 --type sft
!python colab/train_sft.py --round 2 --prev-checkpoint checkpoints/sft/round_1/final

## Round 3 — Attack SFT Round 2

In [ ]:
# Cell 10 — Round 3: Generate attacks
!python scripts/02_run_attack_round.py --round 3 --phase generate

In [ ]:
# Cell 11 — Round 3: Inference (SFT round 2 checkpoint, BASE prompt)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_2/final --seed 42 \
    --input results/attacks/round_3/prompts.jsonl \
    --output results/attacks/round_3/responses.jsonl

In [ ]:
# Cell 12 — Round 3: Judge + ASR
!python scripts/02_run_attack_round.py --round 3 --phase judge

from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_3/results.jsonl")
print(f"\nRound 3 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

In [ ]:
# Cell 13 — Final SFT (round 3) + DPO
!python scripts/03_build_defense_data.py --rounds 1,2,3 --type sft
!python colab/train_sft.py --round 3 --prev-checkpoint checkpoints/sft/round_2/final

!python scripts/03_build_defense_data.py --rounds 1,2,3 --type dpo
!python colab/train_dpo.py --sft-checkpoint checkpoints/sft/round_3/final

## Held-Out Evaluation — 3 Checkpoints x 3 Seeds

In [ ]:
# Cell 14 — Held-out evaluation: all checkpoints x all seeds (BASE prompt throughout)
checkpoints = {
    "base": "base",
    "sft": "checkpoints/sft/round_3/final",
    "dpo": "checkpoints/dpo/final",
}
seeds = [42, 123, 456]

for ckpt_name, ckpt_path in checkpoints.items():
    for seed in seeds:
        out = f"results/eval/{ckpt_name}/seed_{seed}/held_out.jsonl"
        print(f"\n{'='*60}")
        print(f"Running: {ckpt_name} seed={seed}")
        !python colab/run_inference.py \
            --checkpoint {ckpt_path} --seed {seed} \
            --input data/splits/held_out_eval.jsonl \
            --output {out}

In [ ]:
# Cell 15 — Evaluate all checkpoints (McNemar, Cohen's h, bootstrap CI)
!python scripts/04_evaluate.py --checkpoints base,sft,dpo --seeds 42,123,456
!python scripts/04_evaluate.py --checkpoints base,sft,dpo --seeds 42,123,456 --judge-helpfulness

## Transfer + Detection + Figures

In [ ]:
# Cell 16 — Transfer evaluation (Mistral-7B, BASE prompt)
!python colab/run_transfer.py --input data/splits/held_out_eval.jsonl --seed 42
!python scripts/04_evaluate.py --judge-transfer

In [ ]:
# Cell 17 — Perplexity detection baseline
!python colab/run_perplexity.py --checkpoint base \
    --input data/splits/held_out_eval.jsonl \
    --english-input data/seeds/raw_seeds.jsonl \
    --output results/analysis/perplexity_results.json
!python scripts/08_detection_analysis.py

In [ ]:
# Cell 18 — Residual analysis + Figures + Cost report
!python scripts/04_evaluate.py --residual-analysis --checkpoint dpo
!python scripts/05_generate_figures.py --results-dir results/eval/
!python scripts/09_cost_report.py

## ASR Dashboard — Re-run anytime to check all scores

In [ ]:
# Cell 19 — ASR Dashboard: all rounds at a glance
from medics.metrics import compute_asr, compute_per_strategy_asr
from medics.utils import load_jsonl
from pathlib import Path

print("=" * 60)
print("  MediCS ASR Dashboard")
print("=" * 60)

# Attack rounds
for r in range(1, 4):
    path = f"results/attacks/round_{r}/results.jsonl"
    if Path(path).exists():
        results = load_jsonl(path)
        asr = compute_asr(results)
        n_harmful = sum(1 for x in results if x.get("judge_label") == "harmful")
        strats = compute_per_strategy_asr(results)
        strat_str = " | ".join(f"{s}:{v:.0%}" for s, v in sorted(strats.items()))
        print(f"\n  Round {r}: ASR {asr:.1%} ({n_harmful}/{len(results)})")
        print(f"    {strat_str}")
    else:
        print(f"\n  Round {r}: not yet run")

# Held-out evaluation
print(f"\n{'='*60}")
print("  Held-Out Evaluation (seed=42)")
print("=" * 60)
for ckpt in ["base", "sft", "dpo"]:
    path = f"results/eval/{ckpt}/seed_42/held_out.jsonl"
    if Path(path).exists():
        results = load_jsonl(path)
        asr = compute_asr(results)
        print(f"  {ckpt:>5}: ASR {asr:.1%}")
    else:
        print(f"  {ckpt:>5}: not yet run")

# Transfer
transfer_path = "results/transfer/mistral_results.jsonl"
if Path(transfer_path).exists():
    results = load_jsonl(transfer_path)
    asr = compute_asr(results)
    print(f"\n  Transfer (Mistral-7B): ASR {asr:.1%}")